In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

# =============================
# Load YOLOv8 Detector
# =============================
model = YOLO("yolov8n.pt")  # detector

# =============================
# Hook Feature Map (conv layers)
# =============================
target_layers = [0, 2, 4]  # conv layers index
feature_maps = {}

def hook_fmap(module, input, output):
    feature_maps[module] = output.detach()

for i, layer in enumerate(model.model.model):
    if i in target_layers:
        layer.register_forward_hook(hook_fmap)

# =============================
# Hook Dense-like Activation (2 layers)
# =============================
# เลือก conv layer ก่อน head 2 layer
dense_like_layers = [model.model.model[-3], model.model.model[-2]]  # workaround
dense_activations = {}

def hook_dense(module, input, output):
    dense_activations[module] = output.detach()

for layer in dense_like_layers:
    layer.register_forward_hook(hook_dense)

# =============================
# Images
# =============================
image_paths = ["1.JPG", "2.JPG", "3.JPG", "4.JPG", "5.JPG"]

for img_path in image_paths:
    feature_maps.clear()
    dense_activations.clear()

    results = model(img_path)

    num_layers = len(feature_maps)
    fig, axes = plt.subplots(1, num_layers, figsize=(5*num_layers, 5))
    if num_layers == 1:
        axes = [axes]

    for ax, (layer, fmap) in zip(axes, feature_maps.items()):
        fmap = fmap[0]  # batch=0
        fmap_np = fmap.cpu().numpy()
        mean_map = np.mean(fmap_np, axis=0)
        ax.imshow(mean_map, cmap='hot')
        ax.set_title(f"Feature Map: Layer {layer}")
        ax.axis('off')
    plt.suptitle(f"Feature Maps - {img_path}")
    plt.show()
